In [1]:
import pandas as pd
import sqlalchemy as sa
import urllib
from sqlalchemy import create_engine
from sqlalchemy import text
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from bs4 import BeautifulSoup as bs
import requests
import re


In [ ]:
all_games = []

for i in range(1, 25): # Let's start with 5 pages to test
    url = f"https://www.metacritic.com/browse/game/?page={i}"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    
    response = requests.get(url, headers=headers)
    
    # 2. Update the SOUP for every page
    soup = bs(response.text, 'html.parser')
    
    # 3. FIX THE ERROR: Ensure the selector is a STRING inside the parentheses
    items = soup.select('div.relative.max-w-full.flex.flex-col')
    
    print(f"Page {i}: Found {len(items)} games.")

    for item in items:
        # --- TITLE ---
        title_div = item.find('div', attrs={'data-title': True})
        title = title_div.get('data-title') if title_div else "N/A"

        # --- YEAR ---
        year = "N/A"
        for s in item.find_all('span'):
            if re.search(r'\d{4}$', s.get_text(strip=True)):
                year = s.get_text(strip=True)[-4:]
                break

        # --- SCORE ---
        score_div = item.find('div', class_='c-siteReviewScore')
        score = score_div.get_text(strip=True) if score_div else "N/A"

        # --- RATING ---
        rating = "N/A"
        rated_label = item.find('span', string=re.compile("Rated"))
        if rated_label and rated_label.parent:
            rating = rated_label.parent.get_text(strip=True).replace("Rated", "").strip()

        if title != "N/A":
            all_games.append({
                'title': title,
                'year': year,
                'score': score,
                'rating': rating
            })
   

Page 1: Found 24 games.
Page 2: Found 24 games.
Page 3: Found 24 games.
Page 4: Found 24 games.
Page 5: Found 24 games.
Page 6: Found 24 games.
Page 7: Found 24 games.
Page 8: Found 24 games.
Page 9: Found 24 games.
Page 10: Found 24 games.
Page 11: Found 24 games.
Page 12: Found 24 games.
Page 13: Found 24 games.
Page 14: Found 24 games.
Page 15: Found 24 games.
Page 16: Found 24 games.
Page 17: Found 24 games.
Page 18: Found 24 games.
Page 19: Found 24 games.
Page 20: Found 24 games.
Page 21: Found 24 games.
Page 22: Found 24 games.
Page 23: Found 24 games.
Page 24: Found 24 games.
Page 25: Found 24 games.
Page 26: Found 24 games.
Page 27: Found 24 games.
Page 28: Found 24 games.
Page 29: Found 24 games.
Page 30: Found 24 games.
Page 31: Found 24 games.
Page 32: Found 24 games.
Page 33: Found 24 games.
Page 34: Found 24 games.
Page 35: Found 24 games.
Page 36: Found 24 games.
Page 37: Found 24 games.
Page 38: Found 24 games.
Page 39: Found 24 games.
Page 40: Found 24 games.
Page 41: 

In [ ]:
user_score_games = []
for i in range(1, 25):
    url = f"https://www.metacritic.com/browse/game/all/all/all-time/userscore/?page={i}/"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    
    response = requests.get(url, headers=headers)
    
    # 2. Update the SOUP for every page
    soup = bs(response.text, 'html.parser')
    
    # 3. FIX THE ERROR: Ensure the selector is a STRING inside the parentheses
    items = soup.select('div.relative.max-w-full.flex.flex-col')
    print(f"Page {i}: Found {len(items)} games.")

    for item in items:
        # --- TITLE ---
        title_div = item.find('div', attrs={'data-title': True})
        title = title_div.get('data-title') if title_div else "N/A"

        # --- SCORE ---
        user_score_div = item.find('div', class_='c-siteReviewScore')
        user_score = user_score_div.get_text(strip=True) if score_div else "N/A"

        if title != "N/A":
            user_score_games.append({
                'title': title,
                'user_score': user_score,
            })

Page 1: Found 24 games.
Page 2: Found 24 games.
Page 3: Found 24 games.
Page 4: Found 24 games.
Page 5: Found 24 games.
Page 6: Found 24 games.
Page 7: Found 24 games.
Page 8: Found 24 games.
Page 9: Found 24 games.
Page 10: Found 24 games.
Page 11: Found 24 games.
Page 12: Found 24 games.
Page 13: Found 24 games.
Page 14: Found 24 games.
Page 15: Found 24 games.
Page 16: Found 24 games.
Page 17: Found 24 games.
Page 18: Found 24 games.
Page 19: Found 24 games.
Page 20: Found 24 games.
Page 21: Found 24 games.
Page 22: Found 24 games.
Page 23: Found 24 games.
Page 24: Found 24 games.
Page 25: Found 24 games.
Page 26: Found 24 games.
Page 27: Found 24 games.
Page 28: Found 24 games.
Page 29: Found 24 games.
Page 30: Found 24 games.
Page 31: Found 24 games.
Page 32: Found 24 games.
Page 33: Found 24 games.
Page 34: Found 24 games.
Page 35: Found 24 games.
Page 36: Found 24 games.
Page 37: Found 24 games.
Page 38: Found 24 games.
Page 39: Found 24 games.
Page 40: Found 24 games.
Page 41: 

In [4]:
df_meta = pd.DataFrame(all_games)
df_user = pd.DataFrame(user_score_games)
df_user=df_user.drop_duplicates()
df_meta=df_meta.drop_duplicates()
df_games= pd.merge(df_meta, df_user, on='title', how='inner')








In [5]:
df_genre = df_games[['title']].copy()
df_genre['title'] = df_genre['title'].str.replace('[ :]', '-', regex=True)

# 2. Collapse ANY number of consecutive dashes into just one
df_genre['title'] = df_genre['title'].str.replace(r'-+', '-', regex=True)
df_genre['title'] = df_genre['title'].str.replace("'", '')
# 3. Optional: Strip dashes from the start or end
df_genre['title'] = df_genre['title'].str.replace(r'\(.*?\)', '', regex=True)

df_genre['title'] = df_genre['title'].str.strip('-')
df_genre['title'] = df_genre['title'].str.lower()




In [6]:
game_details = []
for i, title in enumerate(df_genre['title']):
    

    url = f'https://www.metacritic.com/game/{title}/details'
    print(url)
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    
    response = requests.get(url, headers=headers)
    
    # 2. Update the SOUP for every page
    soup = bs(response.text, 'html.parser')
    
    # 3. FIX THE ERROR: Ensure the selector is a STRING inside the parentheses
    items = soup.select('div.product-subpage__content')
    
    print(f"Game {i} Found")
    for i, item in enumerate(items):
    # Print the first item's HTML to see why the find() is failing
        
    # 1. Locate the container
        title_div = item.find('div', class_='subpage-header__navigation')
        if title_div:
            Title = title_div.get_text(strip=True)
            print(f"Clean Title: {Title}")
        else:
            Title = "N/A"

        dev_div= item.find('a', class_='c-product-detail-link')
        if dev_div:
            Dev = dev_div.get_text(strip=True)
            print(Dev)
        else:
            Dev = 'N/A'

        genre_div = item.find('span', class_='global-link-button__label')
        if genre_div:
                Genre = genre_div.get_text(strip=True)
                print(Genre)
        else:
             Genre = 'N/A' 
        
        platform_div = item.find('div', class_='c-product-details__section c-product-details__section--grouped')
        if platform_div:
                Platform = platform_div.get_text(strip=True)
                print(Platform)
        else:
            Platform = 'N/A' 
             
        game_details.append({
                'title': Title,
                'dev' : Dev,
                'genre': Genre,
                'platform': Platform
            })


https://www.metacritic.com/game/the-legend-of-zelda-ocarina-of-time/details
Game 0 Found
Clean Title: The Legend of Zelda: Ocarina of Time
Nintendo
Open-World Action
Platforms:Nintendo 64
https://www.metacritic.com/game/soulcalibur/details
Game 1 Found
Clean Title: SoulCalibur
Namco
3D Fighting
Platforms:DreamcastiOS (iPhone/iPad)Xbox 360
https://www.metacritic.com/game/super-mario-galaxy/details
Game 2 Found
Clean Title: Super Mario Galaxy
Nintendo
3D Platformer
Platforms:WiiNintendo Switch
https://www.metacritic.com/game/super-mario-galaxy-2/details
Game 3 Found
Clean Title: Super Mario Galaxy 2
Nintendo
3D Platformer
Platforms:WiiNintendo Switch
https://www.metacritic.com/game/the-legend-of-zelda-breath-of-the-wild/details
Game 4 Found
Clean Title: The Legend of Zelda: Breath of the Wild
Nintendo
Open-World Action
Platforms:Wii UNintendo Switch
https://www.metacritic.com/game/perfect-dark/details
Game 5 Found
https://www.metacritic.com/game/red-dead-redemption-2/details
Game 6 Found

In [7]:
df_game_details = pd.DataFrame(game_details)
df_game_details = df_game_details.drop_duplicates()
df_game_details['platform']=df_game_details['platform'].str.replace('Platforms:', '')
game_consoles = [
    "Atari 2600",
    "Nintendo Entertainment System",
    "Sega Genesis",
    "Super Nintendo",
    "PlayStation",
    "Nintendo 64",
    "Dreamcast",
    "PlayStation 2",
    "Xbox",
    "Nintendo GameCube",
    "Xbox 360",
    "PlayStation 3",
    "Wii",
    "PlayStation 4",
    "Xbox One",
    "Wii U",
    "Nintendo Switch",
    "PlayStation 5",
    "Xbox Series X",
    "Xbox Series S",
    "PC",
    "iOS (iPhone/iPad)",
    "3DS",
    "Game Boy Advance"
]
# 1. Sort by length (Longest first is CRITICAL so 'PlayStation 5' matches before 'PlayStation')
game_consoles.sort(key=len, reverse=True)

pattern = '|'.join([re.escape(console) for console in game_consoles])

# 3. Pre-clean: Remove existing commas and flatten multiple spaces
df_game_details['platform'] = (
    df_game_details['platform']
    .str.replace(',', '', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
)

# 4. Extraction Function
def extract_consoles(text):
    # Find all occurrences of our list items in the text
    # re.IGNORECASE ensures it catches 'playstation' and 'PlayStation'
    matches = re.findall(pattern, text, flags=re.IGNORECASE)
    
    # Use dict.fromkeys to remove duplicates while keeping the original order
    return ", ".join(dict.fromkeys(matches))

# 5. Apply the fix to the column
df_game_details['platform'] = df_game_details['platform'].apply(extract_consoles)

df_games = pd.merge(df_games, df_game_details, on = 'title', how='inner')

df_games.to_csv(r'C:\Users\bryan\OneDrive\Desktop\Python_Projects\Metacritic_Analysis\raw_data\metacritic_data.csv', index=False)
        
